In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub

kagglehub.login()


In [3]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

piotereks_glovec_pl_100d_path = kagglehub.dataset_download('piotereks/glovec-pl-100d')

print('Data source import complete.')


100%|██████████| 715M/715M [00:50<00:00, 14.8MB/s] 

Extracting files...


Data source import complete.


In [ ]:
!pip install fasttext faiss-cpu gensim
!apt-get install -y p7zip-full

In [ ]:
!pip install fasttext faiss-cpu gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.1 MB/s eta 0:00:00:00:0100:01


In [3]:
import numpy as np
import faiss
from gensim.models import KeyedVectors

main_vec_file='/kaggle/input/datasets/piotereks/glovec-pl-100d/glove_100_3_polish.txt'
main_vec_file='datasets/glove_100_3_polish.txt'
# --- 1. Load embeddings ---
wv = KeyedVectors.load_word2vec_format(main_vec_file, binary=False)

# --- 2. Prepare matrix ---
all_words = list(wv.key_to_index.keys())
all_vectors = np.array([wv[w] for w in all_words]).astype('float32')

# --- 3. Normalize once (IMPORTANT) ---
faiss.normalize_L2(all_vectors)

# --- 4. Build FAISS index (exact search) ---
d = all_vectors.shape[1]  # 300
index = faiss.IndexFlatIP(d)  # inner product = cosine (after normalization)
index.add(all_vectors)



In [ ]:
# --- 5. Basic logic for word analogies ---
word_A = "king"
word_B = "queen"
word_C = "bucket"

word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
word_C = "wiadro"


v_A = wv[word_A]
v_B = wv[word_B]
v_C = wv[word_C]

# Vector arithmetic
X_diff = v_B - v_A
v_D_candidate = v_C + X_diff

# Normalize query vector
v_D_candidate = v_D_candidate.astype('float32')
faiss.normalize_L2(v_D_candidate.reshape(1, -1))

# --- 6. Search nearest neighbors ---
k = 25  # Increase k to account for filtered words
scores, indices = index.search(v_D_candidate.reshape(1, -1), k)

# --- 7. Output ---
print(f"Input: {word_C} + ({word_B} - {word_A})")

# Filter out input words (A, B, C) from the results
filtered_results_count = 0
for i in range(k):
    candidate_word = all_words[indices[0][i]]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={scores[0][i]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25: # Display top 5 unique results
        break


Input: mężczyzna + (królowa - król)
1: kobieta (score=0.8573)
2: dziewczyna (score=0.7356)
3: dziewczę (score=0.7256)
4: młoda (score=0.6949)
5: kobiecy (score=0.6783)
6: dziewczynka (score=0.6736)
7: starsza (score=0.6669)
8: blondynka (score=0.6474)
9: jej (score=0.6378)
10: chłopiec (score=0.6330)
11: kate (score=0.6306)
12: pielęgniarka (score=0.6281)
13: jednać (score=0.6275)
14: laura (score=0.6257)
15: wyglądać (score=0.6257)
16: par (score=0.6234)
17: susan (score=0.6232)
18: ruth (score=0.6224)
19: mary (score=0.6208)
20: lucy (score=0.6144)
21: przyjaciółka (score=0.6132)
22: czuły (score=0.6121)
23: podeszły (score=0.6114)
24: była (score=0.6110)


In [7]:
# Probably eighen calculation and space normalisation
import numpy as np
from scipy.linalg import eigh
import time

print("Starting vector processing in cell S7mm7qcZYOTz...")

# ── 1. Load vectors (using pre-loaded wv) ───────────────────────────────────────────
# wv is already loaded by the previous cell (ZxB9XxVWSF6-)
words = list(wv.key_to_index.keys())
X = np.array([wv[w] for w in words]).astype(np.float32)
print(f"Using pre-loaded word vectors (wv) from Gensim. Initial word vectors (X) shape: {X.shape}")

# ── 2. Centre ─────────────────────────────────────────────────
print("Starting centering process...")
t0 = time.time()
mu  = X.mean(axis=0)                  # (dim,)
X_c = X - mu                           # (N, dim) — broadcast
print(f'Centering complete: {time.time()-t0:.2f}s, Centered matrix (X_c) shape: {X_c.shape}')

# ── 3. Covariance  (dim×dim, not N×N) ─────────────────────────
print("Starting covariance calculation...")
t0 = time.time()
cov = (X_c.T @ X_c) / len(X_c)        # (dim, dim)
print(f'Covariance complete: {time.time()-t0:.2f}s, Covariance matrix (cov) shape: {cov.shape}')

# ── 4. Eigendecomposition ─────────────────────────────────────
print("Starting eigendecomposition...")
t0 = time.time()
eigvals, U = eigh(cov)                 # eigvals (dim,), U (dim,dim)
eigvals = eigvals[::-1]               # flip to descending
U       = U[:, ::-1]
eigvals = np.maximum(eigvals, 1e-9)   # regularise near-zeros
print(f'Eigendecomposition complete: {time.time()-t0:.2f}s, Eigenvalues shape: {eigvals.shape}, Eigenvectors (U) shape: {U.shape}')

# ── 5. Build whitening matrix  W = Λ^{-½} Uᵀ ─────────────────
print("Building whitening matrix...")
W = U / np.sqrt(eigvals)               # (dim, dim), cols scaled
W = W.T                                # W shape (dim, dim)
print(f'Whitening matrix (W) built. Shape: {W.shape}')

# ── 6. Transform all vectors  x̃ = W(x−μ) ─────────────────────
print("Transforming all vectors (whitening)...")
t0 = time.time()
X_white = X_c @ W.T                   # (N, dim) — the bottleneck
print(f'Vector transformation complete: {time.time()-t0:.2f}s, Whitened vectors (X_white) shape: {X_white.shape}')

# ── 7. Save ───────────────────────────────────────────────────
print("Saving whitened vectors, whitening matrix, and mean...")
np.save('output-data/X_white.npy',  X_white)      # fast binary
np.save('output-data/W_matrix.npy', W)            # reuse for new words
np.save('output-data/mu.npy',       mu)
with open('output-data/words.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(words))
print("Save complete.")

# ── 8. Query new word at runtime ──────────────────────────────
print("Preparing for runtime queries...")
def whiten_vec(x_raw):
    # Ensure x_raw is a numpy array
    x_raw_np = np.asarray(x_raw, dtype=np.float32)
    return W @ (x_raw_np - mu)            # single O(d²) multiply

def cosine(a, b):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return float(a @ b)

word2idx = {w: i for i, w in enumerate(words)}
# Example usage:
# Check if words 'bank', 'river', 'loan' exist in the vocabulary before querying
test_words = ['bank', 'river', 'loan'] # These are English words, assuming Polish model has them or equivalent
available_test_words = [w for w in test_words if w in word2idx]

if len(available_test_words) >= 3:
    bank_vec  = X_white[word2idx['bank']]
    river_vec = X_white[word2idx['river']]
    loan_vec  = X_white[word2idx['loan']]

    print(f"Cosine similarity between 'bank' and 'river': {cosine(bank_vec, river_vec)}")
    print(f"Cosine similarity between 'bank' and 'loan': {cosine(bank_vec, loan_vec)}")
else:
    print(f"Not all test words ({test_words}) found in vocabulary for similarity calculation. Available words: {list(word2idx.keys())[:10]}...")

print("Cell S7mm7qcZYOTz execution finished.")


Starting vector processing in cell S7mm7qcZYOTz...
Using pre-loaded word vectors (wv) from Gensim. Initial word vectors (X) shape: (1926320, 100)
Starting centering process...
Centering complete: 0.38s, Centered matrix (X_c) shape: (1926320, 100)
Starting covariance calculation...
Covariance complete: 0.45s, Covariance matrix (cov) shape: (100, 100)
Starting eigendecomposition...
Eigendecomposition complete: 0.01s, Eigenvalues shape: (100,), Eigenvectors (U) shape: (100, 100)
Building whitening matrix...
Whitening matrix (W) built. Shape: (100, 100)
Transforming all vectors (whitening)...
Vector transformation complete: 0.19s, Whitened vectors (X_white) shape: (1926320, 100)
Saving whitened vectors, whitening matrix, and mean...
Save complete.
Preparing for runtime queries...
Cosine similarity between 'bank' and 'river': 0.1441093534231186
Cosine similarity between 'bank' and 'loan': 0.272343248128891
Cell S7mm7qcZYOTz execution finished.


In [ ]:
# --- 5. Analogy Logic in Whitened Space --- to be verified if this works
word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
word_C = "wiadro"

# Get index for the words
idx_A = word2idx[word_A]
idx_B = word2idx[word_B]
idx_C = word2idx[word_C]

# Use pre-computed whitened vectors from X_white
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

# Vector arithmetic in whitened space
X_diff_white = v_B_white - v_A_white
v_D_candidate_white = v_C_white + X_diff_white

# --- 6. Search nearest neighbors in X_white ---
# Normalize candidate for cosine similarity
query_vec = v_D_candidate_white / np.linalg.norm(v_D_candidate_white)

# X_white is already normalized from previous cell execution steps if FAISS logic was used,
# but to be safe and independent of index state, we normalize a copy or ensure normalization:
X_white_norm = X_white / np.linalg.norm(X_white, axis=1, keepdims=True)

# Calculate all cosine similarities
similarities = X_white_norm @ query_vec

# Get top indices
k = 25
top_indices = np.argsort(similarities)[::-1]

# --- 7. Output ---
print(f"Input: {word_C} + ({word_B} - {word_A}) [Whitened Space]")

filtered_results_count = 0
for idx in top_indices:
    candidate_word = words[idx]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={similarities[idx]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25:
        break

Input: mężczyzna + (królowa - król) [Whitened Space]
1: kobieta (score=0.8166)
2: dziewczę (score=0.6635)
3: dziewczyna (score=0.6534)
4: khariów (score=0.6318)
5: młoda (score=0.6280)
6: thullijskie (score=0.6010)
7: wierzace (score=0.6004)
8: kobiecy (score=0.5999)
9: starsza (score=0.5991)
10: dziewczynka (score=0.5960)
11: psychloskie (score=0.5930)
12: dźuvli (score=0.5897)
13: par (score=0.5886)
14: ponadtrzydziestoletnia (score=0.5852)
15: czterdziestokilkuletniej (score=0.5803)
16: blondynka (score=0.5772)
17: cedrowicz (score=0.5767)
18: eldreńskie (score=0.5764)
19: ubrany (score=0.5687)
20: jednać (score=0.5663)
21: nietransseksualistki (score=0.5643)
22: jej (score=0.5635)
23: susan (score=0.5603)
24: podeszły (score=0.5596)
25: męski (score=0.5587)


# All below to verify

In [ ]:
import faiss

# --- 5. Analogy Logic in Whitened Space ---
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

# Get indices
idx_A = word2idx[word_A]
idx_B = word2idx[word_B]
idx_C = word2idx[word_C]

# Use pre-computed whitened vectors
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

# Arithmetic in whitened space
X_diff_white = v_B_white - v_A_white
v_D_candidate_white = v_C_white + X_diff_white

# --- 6. Search nearest neighbors using FAISS ---
# 1. Build an index for the whitened vectors
d_white = X_white.shape[1]
# IndexFlatIP is used for Cosine Similarity on normalized vectors
index_white = faiss.IndexFlatIP(d_white)

# 2. Normalize vectors for Cosine Similarity
X_white_norm = X_white.copy().astype('float32')
faiss.normalize_L2(X_white_norm)
index_white.add(X_white_norm)

# 3. Prepare and normalize the query vector
query_vec = v_D_candidate_white.reshape(1, -1).astype('float32')
faiss.normalize_L2(query_vec)

# 4. Search
k = 25
scores, indices = index_white.search(query_vec, k)

# --- 7. Output ---
print(f"Input: {word_C} + ({word_B} - {word_A}) [Whitened Space via FAISS]")

filtered_results_count = 0
for i in range(k):
    idx = indices[0][i]
    candidate_word = words[idx]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={scores[0][i]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25:
        break

Input: wiadro + (królowa - król) [Whitened Space via FAISS]
1: kubeł (score=0.6029)
2: cebrzyk (score=0.5886)
3: zlew (score=0.5564)
4: balia (score=0.5440)
5: dzbanek (score=0.5347)
6: jasicowa (score=0.5322)
7: wiaderko (score=0.5259)
8: kubełek (score=0.5233)
9: pojemnik (score=0.5207)
10: wietliny (score=0.5173)
11: garnek (score=0.5131)
12: kubek (score=0.5085)
13: wydestylowywania (score=0.5085)
14: woda (score=0.5035)
15: koromysła (score=0.4975)
16: bezkaloryczny (score=0.4965)
17: konew (score=0.4956)
18: brachiczna (score=0.4947)
19: wrzątek (score=0.4927)
20: naczynie (score=0.4918)
21: szklanke (score=0.4871)
22: puszka (score=0.4865)
23: ścierka (score=0.4863)
24: beczka (score=0.4854)


In [ ]:
import faiss
import numpy as np

# --- 5. Analogy Logic in Whitened Space ---
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]

v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

X_diff_white = v_B_white - v_A_white
v_D_candidate_white = v_C_white + X_diff_white

# --- 6. Advanced FAISS Search (IVF Algorithm) ---
print("Configuring FAISS IndexIVFFlat (Partitioned KNN search)...")
d_white = X_white.shape[1]

# 1. Normalization
X_white_norm = X_white.copy().astype('float32')
faiss.normalize_L2(X_white_norm)

# 2. Define the algorithm: IndexIVFFlat (Voronoi partitions)
# nlist is the number of clusters (cells)
nlist = int(np.sqrt(X_white_norm.shape[0]))
quantizer = faiss.IndexFlatIP(d_white) # Coarse quantizer
index_ivf = faiss.IndexIVFFlat(quantizer, d_white, nlist, faiss.METRIC_INNER_PRODUCT)

# 3. Train the index (Required for IVF algorithms)
print(f"Training IVF index on {X_white_norm.shape[0]} vectors with {nlist} clusters...")
index_ivf.train(X_white_norm)
index_ivf.add(X_white_norm)

# 4. Set nprobe (How many neighboring clusters to search)
index_ivf.nprobe = 10

# 5. Prepare query vector
query_vec = v_D_candidate_white.reshape(1, -1).astype('float32')
faiss.normalize_L2(query_vec)

# 6. Execute KNN search algorithm
k = 25
scores, indices = index_ivf.search(query_vec, k)

# --- 7. Output ---
print(f"\nInput: {word_C} + ({word_B} - {word_A}) [Whitened Space via IVF-KNN]")
filtered_results_count = 0
for i in range(k):
    idx = indices[0][i]
    if idx == -1: continue # Skip if search failed to find enough neighbors
    candidate_word = words[idx]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={scores[0][i]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25:
        break

Configuring FAISS IndexIVFFlat (Partitioned KNN search)...
Training IVF index on 1926320 vectors with 1387 clusters...

Input: wiadro + (królowa - król) [Whitened Space via IVF-KNN]
1: kubeł (score=0.6029)
2: cebrzyk (score=0.5886)
3: zlew (score=0.5564)
4: balia (score=0.5440)
5: dzbanek (score=0.5347)
6: jasicowa (score=0.5322)
7: wiaderko (score=0.5259)
8: kubełek (score=0.5233)
9: pojemnik (score=0.5207)
10: wietliny (score=0.5173)
11: garnek (score=0.5131)
12: kubek (score=0.5085)
13: wydestylowywania (score=0.5085)
14: woda (score=0.5035)
15: koromysła (score=0.4975)
16: bezkaloryczny (score=0.4965)
17: konew (score=0.4956)
18: brachiczna (score=0.4947)
19: wrzątek (score=0.4927)
20: naczynie (score=0.4918)
21: szklanke (score=0.4871)
22: puszka (score=0.4865)
23: ścierka (score=0.4863)
24: beczka (score=0.4854)


In [ ]:
import faiss
import numpy as np

# --- 5. Analogy Logic in Whitened Space ---
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

# Retrieve indices from word2idx created in the whitening cell
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]

# Select the whitened vectors
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

# Compute the target candidate vector (C + B - A)
X_diff_white = v_B_white - v_A_white
v_D_candidate_white = v_C_white + X_diff_white

# --- 6. FAISS K-Nearest Neighbors Search ---
print("Searching for closest neighbors using FAISS...")
d_white = X_white.shape[1]

# We use IndexFlatIP (Inner Product) because it's equivalent to Cosine Similarity
# when the vectors are L2-normalized.
index_white = faiss.IndexFlatIP(d_white)

# Normalize all vectors to unit length
X_white_norm = X_white.copy().astype('float32')
faiss.normalize_L2(X_white_norm)

# Add vectors to the FAISS index
index_white.add(X_white_norm)

# Prepare the query: normalize the candidate vector
query_vec = v_D_candidate_white.reshape(1, -1).astype('float32')
faiss.normalize_L2(query_vec)

# Search for the k closest neighbors
k = 25
scores, indices = index_white.search(query_vec, k)

# --- 7. Results ---
print(f"\nInput analogy: {word_C} + ({word_B} - {word_A})")
print("Closest neighbors (excluding inputs):")

filtered_results_count = 0
for i in range(k):
    idx = indices[0][i]
    candidate_word = words[idx]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={scores[0][i]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25:
        break

Searching for closest neighbors using FAISS...

Input analogy: wiadro + (królowa - król)
Closest neighbors (excluding inputs):
1: kubeł (score=0.6029)
2: cebrzyk (score=0.5886)
3: zlew (score=0.5564)
4: balia (score=0.5440)
5: dzbanek (score=0.5347)
6: jasicowa (score=0.5322)
7: wiaderko (score=0.5259)
8: kubełek (score=0.5233)
9: pojemnik (score=0.5207)
10: wietliny (score=0.5173)
11: garnek (score=0.5131)
12: kubek (score=0.5085)
13: wydestylowywania (score=0.5085)
14: woda (score=0.5035)
15: koromysła (score=0.4975)
16: bezkaloryczny (score=0.4965)
17: konew (score=0.4956)
18: brachiczna (score=0.4947)
19: wrzątek (score=0.4927)
20: naczynie (score=0.4918)
21: szklanke (score=0.4871)
22: puszka (score=0.4865)
23: ścierka (score=0.4863)
24: beczka (score=0.4854)


In [ ]:
import numpy as np

# --- Parameters ---
# Using a fixed numeric value as requested
margin_value = 1
word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"

# --- 1. Prepare candidate ---
# Using variables from the previous whitening/index cells
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

# Compute the target candidate vector (C + B - A)
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Define the Hypercube Bounds ---
# The cube is +/- 0.01 absolute numeric value per dimension
lower_bounds = v_D_candidate_white - margin_value
upper_bounds = v_D_candidate_white + margin_value

print(f"Searching for words inside a {X_white.shape[1]}D-cube (!{margin_value}) around the candidate...")

# --- 3. Find vectors within bounds ---
# Check which vectors in X_white satisfy lower <= value <= upper for ALL dimensions
mask = np.all((X_white >= lower_bounds) & (X_white <= upper_bounds), axis=1)

# Get indices and words
inside_indices = np.where(mask)[0]
results = [(words[i], i) for i in inside_indices if words[i] not in [word_A, word_B, word_C]]

# --- 4. Display Results ---
print(f"Total words found inside the hypercube: {len(results)}")
print("--------------------------------------------------")
for i, (word, idx) in enumerate(results):
    # Calculate Euclidean distance for reference
    dist = np.linalg.norm(X_white[idx] - v_D_candidate_white)
    print(f"{i+1}: {word} (Euclidean distance to center: {dist:.4f})")

if len(results) == 0:
    print(f"No words found within the !{margin_value} range.")

Searching for words inside a 100D-cube (!1) around the candidate...
Total words found inside the hypercube: 0
--------------------------------------------------
No words found within the !1 range.


In [ ]:
# Diagnostic Cell: Check data alignment and range
import numpy as np

print(f"Shape of X_white: {X_white.shape}")
print(f"Number of words in word2idx: {len(word2idx)}")

# Check a specific word's whitened vector
try:
    test_word = 'król'
    idx = word2idx[test_word]
    vec = X_white[idx]
    print(f"\nWhitened vector for '{test_word}' (first 5 dims): {vec[:5]}")
    print(f"Mean of this vector: {np.mean(vec):.4f}")
    print(f"Std Dev of this vector: {np.std(vec):.4f}")

    # Check the range of the candidate vector
    v_D = v_C_white + (v_B_white - v_A_white)
    print(f"\nCandidate D (first 5 dims): {v_D[:5]}")

    # Check min/max across all vectors to understand the space scale
    print(f"Global Min in X_white: {np.min(X_white):.4f}")
    print(f"Global Max in X_white: {np.max(X_white):.4f}")

    # Debug the mask for a large margin
    temp_margin = 1.0
    low, high = v_D - temp_margin, v_D + temp_margin
    mask_test = np.all((X_white >= low) & (X_white <= high), axis=1)
    print(f"\nWords found with margin 1.0: {np.sum(mask_test)}")

except Exception as e:
    print(f"Error during diagnostics: {e}")

Shape of X_white: (1926320, 100)
Number of words in word2idx: 1926320

Whitened vector for 'król' (first 5 dims): [-7.2682567  -0.25737187 -2.8894212   1.7560194   3.7755213 ]
Mean of this vector: 0.0247
Std Dev of this vector: 2.5325

Candidate D (first 5 dims): [-6.3898926  -0.5971104   1.365499   -1.2411691  -0.16142559]
Global Min in X_white: -14.5455
Global Max in X_white: 16.4090

Words found with margin 1.0: 0
